# Static Move Prediction Draft

## Setup Environment && Load Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
from dataclasses import dataclass

import mlflow
import optuna

from sklearn.ensemble import HistGradientBoostingClassifier

from chesswinnerprediction.static_move.models.static_move_base import (
    StaticMoveBaseModel,
)

from config import RANDOM_STATE, ROOT_DIR
from chesswinnerprediction.static_move.utils import (
    load_train_valid_test,
    setup_mlflow,
    log_prediction,
)

## Setup MLflow

In [3]:
setup_mlflow(experiment_name="Static Move Prediction")

## Load Data

In [4]:
X_train, y_train, X_valid, y_valid, X_test, y_test = load_train_valid_test(
    drop_event=True
)

train_sample_weight = X_train["sample_weight"]
X_train.drop(columns=["sample_weight"], inplace=True)

valid_sample_weight = X_valid["sample_weight"]
X_valid.drop(columns=["sample_weight"], inplace=True)

In [5]:
X_train.head(10)

,eval,EloDiff,MeanElo,BaseTime,IncrementTime,time_diff_norm,white_remaining_time_norm,black_remaining_time_norm,GameDurations_norm,is_checkmate_countdown,i_move,w_score,b_score,n_pieces,white_time_per_move,black_time_per_move,white_increment_pct_in_time_per_move,black_increment_pct_in_time_per_move,white_time_will_end_on_move,black_time_will_end_on_move
0,6.72,-41,1580.5,300,0,-0.323333,0.256667,0.580000,1.163333,False,44,35,34,27,0.016895,0.009546,0.0,0.0000,15.191926,60.755540
1,0.10,18,1572.0,300,0,-0.013333,0.910000,0.923333,0.166667,False,23,38,38,30,0.003914,0.003334,0.0,0.0000,200.000000,200.000000
2,-1.75,-60,1857.0,600,0,0.183333,0.261667,0.078333,1.660000,False,98,16,14,12,0.007535,0.009406,0.0,0.0000,34.726768,8.328228
3,0.31,-32,1485.0,180,2,0.144444,0.277778,0.133333,2.000000,False,122,7,6,11,0.005921,0.007105,1.0,1.0000,46.915152,18.766589
4,8.44,-35,1384.5,300,0,0.373333,0.520000,0.146667,1.333333,False,61,28,22,20,0.007870,0.013990,0.0,0.0000,66.074936,10.483626
5,0.15,-279,1333.5,300,8,0.093333,1.000000,1.000000,0.393333,False,21,33,31,24,0.000001,0.000001,1.0,1.0000,200.000000,200.000000
6,0.66,-7,1534.5,300,0,0.000000,0.986667,0.986667,0.026667,False,7,38,38,30,0.001906,0.001906,0.0,0.0000,200.000000,200.000000
7,0.37,276,1526.0,600,3,0.033333,0.998333,0.965000,0.061667,False,7,39,39,32,0.000239,0.005001,1.0,0.9998,200.000000,192.961408
8,-0.73,-284,1297.0,600,10,-0.043333,0.955000,0.998333,0.430000,False,25,26,26,26,0.001801,0.000068,1.0,1.0000,200.000000,200.000000
9,0.00,-68,1697.0,900,3,0.210000,0.752222,0.542222,1.062222,False,109,0,1,3,0.002274,0.004201,1.0,0.7935,200.000000,129.076062


## Create Interaction Constraints

In [6]:
eval_and_is_checkmate = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("eval"),
    X_train.columns.get_loc("is_checkmate_countdown"),
}
board_features = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("w_score"),
    X_train.columns.get_loc("b_score"),
    X_train.columns.get_loc("n_pieces"),
}
eval_and_board = eval_and_is_checkmate.union(board_features)

time_control = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("GameDurations_norm"),
    X_train.columns.get_loc("black_remaining_time_norm"),
    X_train.columns.get_loc("white_remaining_time_norm"),
    X_train.columns.get_loc("time_diff_norm"),
    X_train.columns.get_loc("IncrementTime"),
    X_train.columns.get_loc("BaseTime"),
}

time_control_extra = {
    X_train.columns.get_loc("black_time_will_end_on_move"),
    X_train.columns.get_loc("white_time_will_end_on_move"),
    X_train.columns.get_loc("black_increment_pct_in_time_per_move"),
    X_train.columns.get_loc("black_increment_pct_in_time_per_move"),
    X_train.columns.get_loc("black_time_per_move"),
    X_train.columns.get_loc("white_time_per_move"),
    X_train.columns.get_loc("i_move"),
}

all_features = set(range(X_train.shape[1]))
other_features = all_features.difference(
    eval_and_board, time_control, time_control_extra
)

interaction_cst = [
    eval_and_is_checkmate,
    board_features,
    eval_and_board,
    time_control,
    time_control_extra,
    other_features,
]

## Optuna

In [7]:
def get_model(trial: optuna.Trial, model_type, hist_gb_class_weight=None):
    if model_type == HistGradientBoostingClassifier.__name__:
        model = HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            tol=1e-5,
            n_iter_no_change=10,
            learning_rate=trial.suggest_float(
                "hist_learning_rate", 5e-4, 1e-2, log=True
            ),
            max_iter=trial.suggest_int("hist_max_iter", 800, 1600),
            max_depth=trial.suggest_int("hist_max_depth", 5, 20),
            min_samples_leaf=trial.suggest_int(
                "hist_min_samples_leaf", 10, X_train.shape[0] // 8
            ),
            max_leaf_nodes=trial.suggest_int("hist_max_leaf_nodes", 50, 300),
            max_bins=trial.suggest_int("hist_max_bins", 100, 255),
            interaction_cst=interaction_cst,
            max_features=trial.suggest_float("hist_max_features", 0.5, 1.0),
            class_weight=hist_gb_class_weight,
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model

In [8]:
@dataclass
class BestModel:
    trial: optuna.Trial = None
    score: float = 0.0
    run_id: str = ""
    model: StaticMoveBaseModel = None

In [11]:
class Objective:
    def __init__(
        self,
        parent_run_id,
        model_types,
        fit_kwargs,
        get_model_kwargs,
        score_model_kwargs,
    ):
        self.parent_run_id = parent_run_id
        self.model_types = model_types
        self.fit_kwargs = fit_kwargs
        self.get_model_kwargs = get_model_kwargs
        self.score_model_kwargs = score_model_kwargs
        self._best_models = self.__generate_best_model_dict()

    def __call__(self, trial: optuna.Trial):
        model_type = trial.suggest_categorical("model_type", self.model_types)

        estimator = get_model(trial, model_type, **self.get_model_kwargs)
        model = StaticMoveBaseModel(estimator=estimator)
        fit_kwargs = self.fit_kwargs[model_type]
        model.fit(X_train, y_train, **fit_kwargs)

        model.score(X_valid, y_valid)
        self.log_trial(trial, model)

        return model.balanced_accuracy

    def log_trial(self, trial, model: StaticMoveBaseModel):
        score = model.balanced_accuracy
        model_type = trial.params["model_type"]
        if score > self._best_models[model_type].score:
            self._best_models[model_type].model = model
            self._best_models[model_type].score = score
            self._best_models[model_type].trial = trial

        run_id = self._best_models[model_type].run_id
        with mlflow.start_run(
            nested=True, run_id=run_id, parent_run_id=self.parent_run_id
        ) as model_run:
            with mlflow.start_run(
                nested=True,
                run_name=f"Trial_{trial.number}",
                parent_run_id=model_run.info.run_id,
            ):
                model.log_trial()

    def log_best(self, trial):
        best_mode = self._best_models[trial.params["model_type"]]
        mlflow.log_params(best_mode.trial.params)
        best_mode.model.log_trial()

        for model_type in self.model_types:
            best_model_data = self._best_models[model_type]
            if best_model_data.model is None:
                mlflow.delete_run(best_model_data.run_id)
                continue
            with mlflow.start_run(
                nested=True,
                run_id=best_model_data.run_id,
                parent_run_id=self.parent_run_id,
            ):
                mlflow.log_params(best_model_data.trial.params)
                best_model_data.model.log_trial()

    def log_res_prediction(self, trial, x, y, set_name):
        model = self._best_models[trial.params["model_type"]].model
        log_prediction(model, x, y, set_name)

    def __generate_best_model_dict(self):
        best_models = {}
        for model_type in self.model_types:
            with mlflow.start_run(
                nested=True, run_name=model_type, parent_run_id=self.parent_run_id
            ) as model_run:
                best_models[model_type] = BestModel(run_id=model_run.info.run_id)
        return best_models

## Study Run

In [12]:
def study_run(description, study_name, objective_kwargs):
    study = optuna.create_study(direction="maximize", study_name=study_name)

    with mlflow.start_run(run_name=study_name, description=description) as main_run:
        objective = Objective(main_run.info.run_id, **objective_kwargs)
        study.optimize(objective, n_trials=64, show_progress_bar=True)
        objective.log_best(study.best_trial)

        objective.log_res_prediction(
            study.best_trial, X_train, y_train, set_name="X_train"
        )
        objective.log_res_prediction(
            study.best_trial, X_valid, y_valid, set_name="X_valid"
        )
        objective.log_res_prediction(
            study.best_trial, X_test, y_test, set_name="X_test"
        )

    return study

#### Try Different Approaches to Class Weights and Sample Weights


In [13]:
models_types = [HistGradientBoostingClassifier.__name__]

study_kwargs_1 = {
    "study_name": "NO Class Weights; NO Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {}},
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": None},
    },
}
study_kwargs_2 = {
    "study_name": "Class Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {}},
        "get_model_kwargs": {"hist_gb_class_weight": "balanced"},
        "score_model_kwargs": {"sample_weight": None},
    },
}
study_kwargs_3 = {
    "study_name": "Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {
            HistGradientBoostingClassifier.__name__: {
                "sample_weight": train_sample_weight
            }
        },
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": None},
    },
}
study_kwargs_4 = {
    "study_name": "Class Weights && Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {
            HistGradientBoostingClassifier.__name__: {
                "sample_weight": train_sample_weight
            }
        },
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": valid_sample_weight},
    },
}

In [ ]:
run_description = (
    "sample weights; valid with sample_weights; 10 bin; time features added"
)

st = study_run(description=run_description, **study_kwargs_4)

## Best Model

In [11]:
best_model = HistGradientBoostingClassifier(
    interaction_cst=[
        {0, 9, 10},
        {10, 11, 12, 13},
        {0, 9, 10, 11, 12, 13},
        {3, 4, 5, 6, 7, 8, 10},
        {10, 14, 15, 17, 18, 19},
        {16, 1, 2},
    ],
    learning_rate=0.009680768024562138,
    max_bins=151,
    max_depth=9,
    max_features=0.6082373387795799,
    max_iter=1132,
    max_leaf_nodes=51,
    min_samples_leaf=1892,
    random_state=42,
    tol=1e-05,
)

In [22]:
best_model.fit(X_train, y_train, sample_weight=train_sample_weight)

HistGradientBoostingClassifier(interaction_cst=[{0, 9, 10}, {10, 11, 12, 13},
                                                {0, 9, 10, 11, 12, 13},
                                                {3, 4, 5, 6, 7, 8, 10},
                                                {17, 18, 19, 10, 14, 15},
                                                {16, 1, 2}],
                               learning_rate=0.009680768024562138, max_bins=151,
                               max_depth=9, max_features=0.6082373387795799,
                               max_iter=1132, max_leaf_nodes=51,
                               min_samples_leaf=1892, random_state=42,
                               tol=1e-05)

## Save Model

In [27]:
import os
import joblib

In [28]:
_ = joblib.dump(best_model, os.path.join(ROOT_DIR, "models", "static_move.pkl"))